# Benchmark Evaluation

`result/statistics/synthetic_data/` 아래의 각 벤치마크 폴더(`wo_` 로 시작하는 ablation 폴더 제외)에 저장된 합성데이터 `0.npy`~`4.npy` 를 읽어, diffusion 과 **동일한 평가 파이프라인**(generation quality + portfolio VaR·ES)으로 평가하고 CSV 로 저장한다.

- 학습/생성은 각 `modules/<model>/run.py` 가 담당 → 이 노트북은 **결과 평가만** 한다.
- 평가 기준(real)은 `data/indices.csv` 의 test 분할 6차원 로그수익률.
- 산출물: 폴더별 `{tag}_gen_quality.csv`, `{tag}_<seed>_portfolio.csv` + 종합표 `benchmark_summary_*.csv`.

In [1]:
import os
import yaml
import ml_collections
import numpy as np
import pandas as pd
import torch
from utils.utils import *
from utils.metric import *
from utils.eval_portfolio.summary import *
from utils.eval_portfolio.strategies import log_return_to_price
from utils.file_manager import LOCAL_BASE_DIRECTORY, quantDiffusionPath

file_manager = quantDiffusionPath()

# Real data (평가 기준)
`data/indices.csv` 의 test 분할 6차원 로그수익률을 모든 벤치마크 공통 real 로 사용한다. (학습/생성용 scaling 은 불필요 — 평가 reference 만 준비.)

In [2]:
# 평가 기준 real 데이터: data/indices.csv 의 test 분할 6차원 로그수익률
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seq_len = 256

df = pd.read_csv("./data/indices.csv")
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
df.set_index("Date", inplace=True)
df = df.apply(pd.to_numeric).astype(float)

df_log_returns = np.log1p(df.pct_change()).dropna()
train_size = int(0.7 * len(df_log_returns))
test_raw = df_log_returns.iloc[train_size:, :]               # test 구간 (diffusion 과 동일 분할)

# real_data: (B, 6, T) — 모든 벤치마크 공통 평가 기준
test_windowed = torch.from_numpy(rolling_window(test_raw.values, seq_len)).float()
real_data = test_windowed.transpose(1, 2).cpu().numpy()
print(f"🚀 real_data (test 6d): {real_data.shape}")

🚀 real_data (test 6d): (1538, 6, 256)


c:\Users\Juhwankim\miniconda3\envs\research\lib\site-packages\pandas\core\internals\blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)


# 평가 함수
폴더의 `0..4.npy` 를 읽어 generation-quality + portfolio(VaR·ES) 를 평가해 CSV 로 저장하고, 종합표로 모은다.

In [3]:
# ===== 평가 함수 (학습/생성 없음; 저장된 npy 만 평가) =====
N_RUNS = 5


def evaluate_benchmark(tag, real_data, n_runs=N_RUNS):
    """벤치마크 폴더의 0..n_runs-1.npy 를 읽어 gen-quality + portfolio 평가 후 CSV 저장."""
    rp = file_manager.get_result_path(tag)
    windows = [1, 5, 20, 100]

    # ---- generation quality ----
    total_result_df = pd.DataFrame()
    for seed in range(n_runs):
        fake_data = np.load(f"{rp}/{seed}.npy")
        result_df = pd.DataFrame()
        result_df = pd.concat([result_df, calculate_ccf(real_data, fake_data, 0)], axis=0)
        total_CCF_df = pd.DataFrame(columns=["Average"])
        for i in range(1, 11):
            total_CCF_df = pd.concat([total_CCF_df, calculate_ccf(real_data, fake_data, i)], axis=1)
        result_df = pd.concat([result_df, pd.DataFrame(total_CCF_df.mean(axis=1), columns=["Average"])], axis=0)
        result_df = pd.concat([result_df, calculate_distribution_scores(real_data, fake_data, windows)], axis=0)
        result_df = pd.concat([result_df, calculate_skew_kurtosis(real_data, fake_data, windows)], axis=0)
        _, acf_df = calculate_acf_score(real_data, fake_data, 40)
        result_df = pd.concat([result_df, acf_df], axis=0)
        _, leverage_df = calculate_leverage_effect(real_data, fake_data, 40)
        result_df = pd.concat([result_df, leverage_df], axis=0)
        total_result_df = pd.concat([total_result_df, result_df], axis=1)
    statistics_df = pd.DataFrame(index=total_result_df.index, columns=["MEAN", "STD"])
    statistics_df["MEAN"] = total_result_df.mean(axis=1).values
    statistics_df["STD"] = total_result_df.std(axis=1).values
    statistics_df.to_csv(file_manager.get_result_file(f"{tag}_gen_quality"), encoding="utf-8-sig")

    # ---- portfolio (VaR·ES relative error) ----
    config_dir = f"{LOCAL_BASE_DIRECTORY}/utils/eval_portfolio/config.yaml"
    with open(config_dir) as file:
        eval_config = ml_collections.ConfigDict(yaml.safe_load(file))
    alphas = eval_config.alphas
    strat_names = ["equal_weight", "mean_reversion", "trend_following", "vol_trading"]
    strat_map = {"equal_weight": "EW", "mean_reversion": "MR", "trend_following": "TF", "vol_trading": "VT"}
    real_data_torch = torch.from_numpy(real_data).float().transpose(1, 2)   # (Nr, T, 6)
    for seed in range(n_runs):
        fake_data = np.load(f"{rp}/{seed}.npy")
        fake_data_torch = torch.from_numpy(fake_data).float().transpose(1, 2)   # (Nf, T, 6)
        dev = fake_data_torch.device
        Nf = fake_data_torch.shape[0]
        # 초기가격은 100 으로 고정 (상대오차 평가라 절대 수준 무관)
        fake_prices_all = log_return_to_price(
            fake_data_torch, torch.full((Nf, 1, 6), 100.0, device=dev))
        # 가격 경로가 비양수(누적 언더플로/NaN)인 fake 샘플 제외
        valid = (fake_prices_all > 0).reshape(Nf, -1).all(dim=1)
        if int(valid.sum()) == 0:
            raise ValueError(f"[{tag}] seed {seed}: 모든 fake 가격이 비양수 — 생성 품질 확인 필요")
        if int(valid.sum()) < Nf:
            print(f"⚠️ [{tag}] seed {seed}: 비양수 가격 {Nf - int(valid.sum())}개 제외")
        fake_prices_all = fake_prices_all[valid]
        # EvaluationComponent 는 real/fake 를 같은 idx 로 인덱싱하므로 개수를 맞춤
        n_samples = min(fake_prices_all.shape[0], real_data.shape[0])
        fake_prices = fake_prices_all[:n_samples]
        real_prices = log_return_to_price(
            real_data_torch[:n_samples],
            torch.full((n_samples, 1, 6), 100.0, device=real_data_torch.device))
        final_results = {}
        with torch.no_grad():
            for alpha in alphas:
                eval_config.Evaluation.TestMetrics.var.alpha = alpha
                eval_config.Evaluation.TestMetrics.es.alpha = alpha
                var_row, es_row = {}, {}
                for strat_name in strat_names:
                    subres_dict, _, _ = full_evaluation(fake_prices, real_prices, eval_config, strat_name=strat_name)
                    sn = strat_map[strat_name]
                    var_row[sn] = round(subres_dict.get("var_rel_mean", 0.0), 4)
                    es_row[sn] = round(subres_dict.get("es_rel_mean", 0.0), 4)
                var_row["Average"] = round(np.mean(list(var_row.values())), 4)
                es_row["Average"] = round(np.mean(list(es_row.values())), 4)
                final_results[f"RE_VaR({alpha})"] = var_row
                final_results[f"RE_ES({alpha})"] = es_row
        ordered_keys = [f"{m}({a})" for m in ["RE_VaR", "RE_ES"] for a in alphas]
        df_final = pd.DataFrame.from_dict(final_results, orient="index")
        cols_order = [strat_map[s] for s in strat_names] + ["Average"]
        df_final = df_final.reindex(index=ordered_keys, columns=cols_order)
        df_final.index.name = "Metric"
        df_final.T.to_csv(file_manager.get_result_file(f"{tag}_{seed}_portfolio"), encoding="utf-8-sig")
    print(f"✅ [{tag}] 저장: {tag}_gen_quality.csv, {tag}_<seed>_portfolio.csv")


def _pm(m, s, nd=4):
    return f"{m:.{nd}f} ± {s:.{nd}f}" if (np.isfinite(m) and np.isfinite(s)) else ""


def aggregate_gen_quality(tag_label_map, out_name="benchmark_summary_gen_quality"):
    cols = {}
    for tag, label in tag_label_map.items():
        f = file_manager.get_result_file(f"{tag}_gen_quality")
        if not os.path.exists(f):
            print(f"  (skip {tag}: {os.path.basename(f)} 없음)")
            continue
        df = pd.read_csv(f, index_col=0)
        cols[label] = pd.Series([_pm(m, s) for m, s in zip(df["MEAN"], df["STD"])], index=df.index)
    summary = pd.DataFrame(cols)
    summary.to_csv(file_manager.get_result_file(out_name), encoding="utf-8-sig")
    print("saved:", file_manager.get_result_file(out_name))
    return summary


def aggregate_portfolio(tag_label_map, out_name="benchmark_summary_portfolio", n_runs=N_RUNS):
    cols = {}
    for tag, label in tag_label_map.items():
        parts = []
        for seed in range(n_runs):
            f = file_manager.get_result_file(f"{tag}_{seed}_portfolio")
            if os.path.exists(f):
                parts.append(pd.read_csv(f, index_col=0))
        if not parts:
            print(f"  (skip {tag}: portfolio csv 없음)")
            continue
        stacked = pd.concat(parts, axis=0)
        avg_rows = stacked.loc[["Average"]]
        m, sd = avg_rows.mean(axis=0), avg_rows.std(axis=0)
        cols[label] = pd.Series([_pm(a, b) for a, b in zip(m, sd)], index=m.index)
    summary = pd.DataFrame(cols)
    summary.to_csv(file_manager.get_result_file(out_name), encoding="utf-8-sig")
    print("saved:", file_manager.get_result_file(out_name))
    return summary

# 평가 실행 + 종합 테이블
`synthetic_data/` 의 비-`wo_` 폴더를 자동으로 찾아 각각 평가(개별 CSV)한 뒤, 종합표(`benchmark_summary_*.csv`)로 모은다.

In [4]:
# ===== 1. 벤치마크 폴더 자동 발견 + 평가 =====
# result/statistics/synthetic_data 아래에서 'wo_' 로 시작하는 ablation 폴더는 제외하고,
# 0..4.npy 가 모두 있는 폴더만 평가 대상으로 삼는다.
base = os.path.join(LOCAL_BASE_DIRECTORY, "result", "statistics", "synthetic_data")
tags = []
for d in sorted(os.listdir(base)):
    if not os.path.isdir(os.path.join(base, d)) or d.startswith("wo_"):
        continue
    if all(os.path.exists(os.path.join(base, d, f"{i}.npy")) for i in range(N_RUNS)):
        tags.append(d)
    else:
        print(f"  (skip {d}: 0..{N_RUNS - 1}.npy 가 모두 있지는 않음)")
print("평가 대상:", tags)

# for tag in tags:
#     print(f"\n{'=' * 70}\n평가: {tag}\n{'=' * 70}")
#     evaluate_benchmark(tag, real_data, n_runs=N_RUNS)

평가 대상: ['diffusionTS', 'quantgan', 'student_t', 'tailgan']


In [5]:
# ===== 2. 종합 테이블 — 발견된 모든 비-wo 벤치마크의 개별 CSV 를 모아 정리 =====
# label 은 폴더명을 그대로 쓰되, 알려진 모델은 PRETTY 로 보기 좋게 덮어쓴다.
# (student_t = QuantDiffusion = ours,  diffusionTS = Diffusion-TS baseline)
PRETTY = {
    "student_t": "QuantDiffusion (ours)",
    "diffusionTS": "Diffusion-TS",
    "quantgan": "QuantGAN",
    "tailgan": "TailGAN",
}
tag_label_map = {t: PRETTY.get(t, t) for t in tags}

# print("[Benchmark 종합 - Generation Quality]")
# summary_gen_quality = aggregate_gen_quality(tag_label_map)
# print(summary_gen_quality)

# print("\n[Benchmark 종합 - Portfolio (Average 전략)]")
# summary_portfolio = aggregate_portfolio(tag_label_map)
# print(summary_portfolio)

# 시각화 (논문용)
비-`wo_` 벤치마크별 **ACF**(`result/images/acf`), **상관 히트맵**(`.../correlation`), **누적수익률 분포 1/5/20/100일**(`.../histogram`)을 모델별로 저장. 추가로 논문용 한 장 — **asset 2(JPM)의 5일 히스토그램**을 4모델(TailGAN→QuantGAN→Diffusion-TS→QuantDiffusion) 한 줄, 투명배경으로 `.../histogram/paper_hist_5day_asset2_JPM.png` 저장.

In [6]:
# ===== 3. 시각화: 비-wo 벤치마크별 ACF/상관/히스토그램 + 논문용 5일 히스토그램 한 장 =====
# real 기준 = test 6d(real_data), 각 모델 합성 = {tag}/0.npy.
# 결과는 result/images 아래 타입별 폴더(acf / correlation / histogram)에 투명배경 PNG 로 저장.
IMG = file_manager.get_image_path()
ACF_DIR = os.path.join(IMG, "acf")
CORR_DIR = os.path.join(IMG, "correlation")
HIST_DIR = os.path.join(IMG, "histogram")
feature_names = ["DJI", "NASDAQ", "JPM", "HSI", "GOLD", "WTI_OIL"]
windows = [1, 5, 20, 100]

# 실데이터 상관 히트맵 (모든 모델 공통) — 1회만 저장
real_list = [real_data[:, i, :] for i in range(real_data.shape[1])]
plot_correlation_heatmap(
    correlation_mean(np.transpose(real_data, (0, 2, 1))),
    "Real", feature_names, out_dir=CORR_DIR, name="real",
)

# 모델별 ACF + 상관 히트맵 + 누적수익률 분포(1/5/20/100일, 자산별)
fakes = {}
for tag in tags:
    fake_data = np.load(os.path.join(file_manager.get_result_path(tag), "0.npy"))
    fakes[tag] = fake_data
    fake_list = [fake_data[:, i, :] for i in range(fake_data.shape[1])]
    plot_correlation_heatmap(
        correlation_mean(np.transpose(fake_data, (0, 2, 1))),
        PRETTY.get(tag, tag), feature_names, out_dir=CORR_DIR, name=tag,
    )
    plot_acf_comparison(real_list, fake_list, len(feature_names), lags=100,
                        out_dir=ACF_DIR, name=tag)
    plot_cumulative_return_dist(real_list, fake_list, windows=windows,
                                n_assets=len(feature_names), out_dir=HIST_DIR, name=tag)
    print(f"[plot] {tag}: ACF -> {ACF_DIR}, correlation -> {CORR_DIR}, histogram -> {HIST_DIR}")

# 논문용 한 장: asset 2(JPM)의 5일 누적수익률 히스토그램, 4모델 한 줄
# 순서: TailGAN -> QuantGAN -> Diffusion-TS -> QuantDiffusion(ours)
paper_order = [t for t in ["tailgan", "quantgan", "diffusionTS", "student_t"] if t in tags]
paper_order += [t for t in tags if t not in paper_order]
plot_hist_row_by_model(
    real_data, {t: fakes[t] for t in paper_order}, PRETTY, window=5, asset_idx=2,
    out_path=os.path.join(HIST_DIR, "paper_hist_5day_asset2_JPM.png"),
)

[plot] diffusionTS: ACF -> c:\Users\Juhwankim\Documents\Github\quantDiffusion\result/images\acf, correlation -> c:\Users\Juhwankim\Documents\Github\quantDiffusion\result/images\correlation, histogram -> c:\Users\Juhwankim\Documents\Github\quantDiffusion\result/images\histogram
[plot] quantgan: ACF -> c:\Users\Juhwankim\Documents\Github\quantDiffusion\result/images\acf, correlation -> c:\Users\Juhwankim\Documents\Github\quantDiffusion\result/images\correlation, histogram -> c:\Users\Juhwankim\Documents\Github\quantDiffusion\result/images\histogram
[plot] student_t: ACF -> c:\Users\Juhwankim\Documents\Github\quantDiffusion\result/images\acf, correlation -> c:\Users\Juhwankim\Documents\Github\quantDiffusion\result/images\correlation, histogram -> c:\Users\Juhwankim\Documents\Github\quantDiffusion\result/images\histogram
[plot] tailgan: ACF -> c:\Users\Juhwankim\Documents\Github\quantDiffusion\result/images\acf, correlation -> c:\Users\Juhwankim\Documents\Github\quantDiffusion\result/image